In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from typing import List, Union, Optional, Tuple

In [2]:
def collect_metrics_dataframe(paths:List[Path], name:str)->pd.DataFrame:
    '''
    Функция принимает на вход список путей до целевых csv файлов с метриками
    и параметр name указывающий название корневого каталога.
    Функция возвращает объединенный DataFrame.
    '''
    all_dfs = []
    
    for path in paths:
        df = pd.read_csv(path)
        parts = path.parts
        if name in parts:
            idx = parts.index(name)
            model_config = "_".join(parts[idx + 1 : -1])
        else:
            print(f'Name {name} not found in {path}')
            
        df.insert(loc=0, column='model_configuration', value=model_config)
        all_dfs.append(df)

    final_df = pd.concat(all_dfs, ignore_index=True)
    return final_df

In [3]:
def plot_metrics_by_target(df: pd.DataFrame,
                           metric_name: str="R2",
                           target: Optional[Union[List[str], str]]=None, 
                           sharey: bool=False, 
                           save_path:Optional[Union[str,Path]]=None
                          ):
    '''
    Визуализирует метрики по моделям и сплитам с фильтрацией по Target (CNi).

    df - DataFrame с исходными данными.
    metric_name - Название метрики ('R2', 'MAE', 'RMSE').
    target - Строка ('CN1'), список строк (['CN1', 'CN2']) или None (все).
    sharey - Зафиксировать ли общую ось Y для графиков.
    '''
    sns.reset_defaults()

    if metric_name not in df.columns:
        raise ValueError(f"Метрика '{metric_name}' не найдена в колонках DataFrame.")

    plot_df = df.copy()
    if target is not None:
        targets_to_plot = [target] if isinstance(target, str) else target
        plot_df = plot_df[plot_df['Target'].isin(targets_to_plot)]

        if plot_df.empty:
            raise ValueError(f"Указанный Target '{target}' не найден в данных.")

    sns.set_theme(style='whitegrid')

    g = sns.catplot(data=plot_df, 
                    x='model_configuration', 
                    y=metric_name, 
                    hue='Split', 
                    col='Target', 
                    kind='bar',
                    col_wrap=4 if len(plot_df['Target'].unique()) > 1 else None,
                    palette="muted",
                    height=5,
                    aspect=1.2,
                    sharey=sharey,
                    legend_out = False,
                )

    g.set_xticklabels(rotation=45, horizontalalignment='right')
    g.set_titles('{col_name}')
    g.set_axis_labels('Модель', metric_name)

    plt.tight_layout()

    if save_path:
        plt.savefig(save_path,dpi=600)
    
    plt.show()

In [4]:
def find_chart_edge(img: np.ndarray, 
                    y_range: Optional[Tuple[int, int]] = None, 
                    x_range: Optional[Tuple[int, int]] = None, 
                    threshold: float = 0.2, 
                    side: str = 'left'
                   ) -> int:
    '''
    Находит X-координату границы графика (левой или правой).
    side - "right" или "left"
    '''
    y_min, y_max = y_range or (0, img.shape[0])
    x_min, x_max = x_range or (0, img.shape[1])

    strip = img[y_min:y_max, x_min:x_max]

    if strip.shape[2] >= 3:
        gray = 0.299 * strip[:, :, 0] + 0.587 * strip[:, :, 1] + 0.114 * strip[:, :, 2]
    else:
        gray = strip[:, :, 0]

    col_min = gray.min(axis=0)

    if side == 'right':
        col_min = col_min[::-1]

    dark_cols = np.where(col_min < threshold)[0]
    if dark_cols.size > 0:
        idx = dark_cols[0]
        return x_max - 1 - idx if side == 'right' else x_min + idx

    return x_max if side == 'right' else x_min

In [5]:
def plot_model_feature_importance(metrics_path: Union[Path, str], 
                                  model_title: str, 
                                  num_cn: int = 7, 
                                  crop_y_top: int = 2620, 
                                  crop_y_bottom: int = 3000, 
                                  crop_x_left: int = 400, 
                                  crop_x_right: int = 6000, 
                                  auto_crop_x: bool = True,
                                  separator_height: int = 15, 
                                  save_path: Optional[Union[str, Path]] = None
                                 ) -> None:
    '''
    Отрисовывает вертикальный стек хитмапов CN1..CNi с подписями слева.
    
    metrics_path - путь к файлу final_metrics.csv или к папке с моделью.
    model_title - заголовок графика (название модели).
    num_cn - количество подпапок CN (по умолчанию 7).
    crop_y_top, crop_y_bottom - вертикальные границы обрезки для обычных картинок.
    crop_x_left, crop_x_right - дефолтные горизонтальные границы обрезки.
    auto_crop_x - автоопределение X-границ с помощью find_chart_edge.
    separator_height - толщина белой разделительной полосы в пикселях.
    save_path - путь для сохранения итоговой картинки.
    '''
    sns.reset_defaults()
    metrics_path = Path(metrics_path)
    model_dir = metrics_path.parent if metrics_path.is_file() else metrics_path

    crops = []
    labels = []
    
    detected_x_left, detected_x_right = crop_x_left, crop_x_right

    for i in range(1, num_cn + 1):
        img_path = Path(model_dir, f'CN{i}', 'feature_importance_map.png')

        if not img_path.exists():
            print(f'Файл не найден: {img_path}')
            continue

        img = plt.imread(img_path)

        if auto_crop_x and not crops:
            # уменьшаем окно по вертикали (двигаем вниз верхнюю границу), чтобы не задевать боковой рисунок.
            detected_x_left = find_chart_edge(img, y_range=(crop_y_top+300, crop_y_bottom), side='left')
            # немного расширяем правую границу, чтобы нормально отображались единицы измерения.
            detected_x_right = find_chart_edge(img, y_range=(crop_y_top+300, crop_y_bottom), side='right') + 50

        # Обрезка по Y и X (последняя картинка берется до самого низа вместе с осью X)
        y_end = crop_y_bottom if i < num_cn else img.shape[0]
        crop = img[crop_y_top:y_end, detected_x_left:detected_x_right]

        crops.append(crop)
        labels.append(f"CN{i}")

    if not crops:
        raise FileNotFoundError(f"Не удалось загрузить ни одного изображения из {model_dir}")

    sep_shape = (separator_height, crops[0].shape[1], crops[0].shape[2])
    white_sep = np.ones(sep_shape, dtype=crops[0].dtype)

    stacked_layers = []
    for idx, crop in enumerate(crops):
        stacked_layers.append(crop)
        if idx < len(crops) - 1:
            stacked_layers.append(white_sep)

    full_image = np.vstack(stacked_layers)

    standard_heatmap_height = crops[0].shape[0]
    y_centers = []
    current_y = 0

    for crop in crops:
        y_centers.append(current_y + standard_heatmap_height / 2)
        current_y += crop.shape[0] + separator_height

    aspect_ratio = full_image.shape[0] / full_image.shape[1]
    fig, ax = plt.subplots(figsize=(16, 16 * aspect_ratio + 1.0))

    ax.imshow(full_image)

    # Настройка подписей CN1..CNi слева по оси Y
    ax.set_yticks(y_centers)
    ax.set_yticklabels(labels, fontsize=13, fontweight='bold')
    ax.tick_params(left=False, bottom=False, labelbottom=False)

    # Убираем рамки вокруг графика
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(model_title, fontsize=16, fontweight='bold', pad=20)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=600)

    plt.show()

In [6]:
def plot_dual_scenario_auto_aligned(path_scenario_a: Union[Path, str],
                                    path_scenario_b: Union[Path, str],
                                    model_title: str,
                                    label_a: str = 'Scenario A',
                                    label_b: str = 'Scenario B',
                                    num_cn: int = 7,
                                    crop_y_top: int = 2650,
                                    crop_y_bottom: int = 2962,
                                    target_width: int = 5700,
                                    separator_height: int = 30,
                                    save_path: Optional[Union[Path, str]] = None
                                   ) -> None:
    '''
    Отрисовывает вертикальный стек пар хитмапов (Сценарий A и Сценарий B) для каждого CNi.
    
    path_scenario_a, path_scenario_b - пути к метрикам/папкам двух сценариев.
    model_title - общий заголовок графика.
    label_a, label_b - подписи сценариев на оси Y.
    num_cn - количество подпапок CN.
    crop_y_top, crop_y_bottom - границы обрезки по Y.
    target_width - фиксированная ширина вырезанной области от левого края.
    separator_height - толщина разделительной белой полосы между блоками CN.
    save_path - путь для сохранения итогового файла.
    '''
    sns.reset_defaults()
    
    path_a = Path(path_scenario_a)
    path_b = Path(path_scenario_b)

    dir_a = path_a.parent if path_a.is_file() else path_a
    dir_b = path_b.parent if path_b.is_file() else path_b

    crops_a = []
    crops_b = []
    valid_cns = []

    for i in range(1, num_cn + 1):
        cn_folder = f'CN{i}'
        img_a_path = Path(dir_a, cn_folder, 'feature_importance_map.png')
        img_b_path = Path(dir_b, cn_folder, 'feature_importance_map.png')

        if not img_a_path.exists() or not img_b_path.exists():
            print(f"Предупреждение: Пропущена папка {cn_folder}")
            continue

        img_a = plt.imread(img_a_path)
        img_b = plt.imread(img_b_path)

        search_y = (crop_y_top + 300, crop_y_bottom)
        x_left_a = max(0, find_chart_edge(img_a, y_range=search_y, side='left') - 50)
        x_left_b = max(0, find_chart_edge(img_b, y_range=search_y, side='left') - 50)

        x_right_a = x_left_a + target_width
        x_right_b = x_left_b + target_width

        y_end_a = crop_y_bottom
        y_end_b = crop_y_bottom if i < num_cn else img_b.shape[0]

        crop_a = img_a[crop_y_top:y_end_a, x_left_a:x_right_a]
        crop_b = img_b[crop_y_top:y_end_b, x_left_b:x_right_b]

        crops_a.append(crop_a)
        crops_b.append(crop_b)
        valid_cns.append(i)

    if not crops_a:
        raise FileNotFoundError('Не удалось загрузить ни одного совпавшего изображения из указанных директорий.')

    sample = crops_a[0]
    sep_shape = (separator_height, sample.shape[1], sample.shape[2])
    white_sep = np.ones(sep_shape, dtype=sample.dtype)

    stacked_layers = []
    y_ticks = []
    y_labels = []
    current_y = 0

    standard_h_a = crops_a[0].shape[0]
    standard_h_b = crops_b[0].shape[0]

    for idx, (crop_a, crop_b, cn_num) in enumerate(zip(crops_a, crops_b, valid_cns)):
        # Склеиваем пару A и B для текущего CN
        stacked_layers.append(crop_a)
        stacked_layers.append(crop_b)

        # Вычисляем Y-координаты подписей для A и B
        center_a = current_y + standard_h_a / 2
        center_b = current_y + crop_a.shape[0] + standard_h_b / 2

        y_ticks.extend([center_a, center_b])
        y_labels.extend([f'CN{cn_num}\n({label_a})', f'CN{cn_num}\n({label_b})'])

        current_y += crop_a.shape[0] + crop_b.shape[0]

        # Добавляем разделитель между разными CN
        if idx < len(crops_a) - 1:
            stacked_layers.append(white_sep)
            current_y += separator_height

    full_image = np.vstack(stacked_layers)

    aspect_ratio = full_image.shape[0] / full_image.shape[1]
    fig, ax = plt.subplots(figsize=(16, 16 * aspect_ratio + 1.2))

    ax.imshow(full_image)

    # Настройка Y-меток слева
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=9, fontweight='bold')
    ax.tick_params(left=False, bottom=False, labelbottom=False)

    # Убираем внешнюю рамку
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(model_title, fontsize=15, fontweight='bold', pad=20)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=600)

    plt.show()

In [7]:
def plot_mean_std_auto_aligned(paths_mean: List[Path],
                               paths_std: List[Path],
                               model_title: str,
                               label_a: str = 'Mean',
                               label_b: str = 'Std',
                               crop_y_top: int = 2650,
                               crop_y_bottom: int = 2962,
                               target_width: int = 5700,
                               separator_height: int = 30,
                               save_path: Optional[Union[Path, str]] = None
                              ) -> None:
    '''
    Отрисовывает вертикальный стек пар хитмапов (Сценарий A и Сценарий B) для каждого CNi.
    
    paths_mean, paths_std - согласованные списки путей к мапам.
    model_title - общий заголовок графика.
    label_a, label_b - подписи сценариев на оси Y.
    crop_y_top, crop_y_bottom - границы обрезки по Y.
    target_width - фиксированная ширина вырезанной области от левого края.
    separator_height - толщина разделительной белой полосы между блоками CN.
    save_path - путь для сохранения итогового файла.
    '''
    sns.reset_defaults()
    
    crops_a = []
    crops_b = []
    valid_cns = []
    num_cn = len(paths_mean)

    for a,b in zip(paths_mean, paths_std):
        cn_i = a.parent.name
        i = int(cn_i[-1])
        img_a_path = Path(a, 'feature_importance_map.png')
        img_b_path = Path(b, 'feature_importance_map.png')

        if not img_a_path.exists() or not img_b_path.exists():
            print(f"Предупреждение: Пропущена папка {cn_i}")
            continue

        img_a = plt.imread(img_a_path)
        img_b = plt.imread(img_b_path)

        search_y = (crop_y_top + 300, crop_y_bottom)
        x_left_a = max(0, find_chart_edge(img_a, y_range=search_y, side='left') - 50)
        x_left_b = max(0, find_chart_edge(img_b, y_range=search_y, side='left') - 50)

        x_right_a = x_left_a + target_width
        x_right_b = x_left_b + target_width

        y_end_a = crop_y_bottom
        y_end_b = crop_y_bottom if i < num_cn else img_b.shape[0]

        crop_a = img_a[crop_y_top:y_end_a, x_left_a:x_right_a]
        crop_b = img_b[crop_y_top:y_end_b, x_left_b:x_right_b]

        crops_a.append(crop_a)
        crops_b.append(crop_b)
        valid_cns.append(i)

    if not crops_a:
        raise FileNotFoundError('Не удалось загрузить ни одного совпавшего изображения из указанных директорий.')

    sample = crops_a[0]
    sep_shape = (separator_height, sample.shape[1], sample.shape[2])
    white_sep = np.ones(sep_shape, dtype=sample.dtype)

    stacked_layers = []
    y_ticks = []
    y_labels = []
    current_y = 0

    standard_h_a = crops_a[0].shape[0]
    standard_h_b = crops_b[0].shape[0]

    for idx, (crop_a, crop_b, cn_num) in enumerate(zip(crops_a, crops_b, valid_cns)):
        # Склеиваем пару A и B для текущего CN
        stacked_layers.append(crop_a)
        stacked_layers.append(crop_b)

        # Вычисляем Y-координаты подписей для A и B
        center_a = current_y + standard_h_a / 2
        center_b = current_y + crop_a.shape[0] + standard_h_b / 2

        y_ticks.extend([center_a, center_b])
        y_labels.extend([f'CN{cn_num}\n({label_a})', f'CN{cn_num}\n({label_b})'])

        current_y += crop_a.shape[0] + crop_b.shape[0]

        # Добавляем разделитель между разными CN
        if idx < len(crops_a) - 1:
            stacked_layers.append(white_sep)
            current_y += separator_height

    full_image = np.vstack(stacked_layers)

    aspect_ratio = full_image.shape[0] / full_image.shape[1]
    fig, ax = plt.subplots(figsize=(16, 16 * aspect_ratio + 1.2))

    ax.imshow(full_image)

    # Настройка Y-меток слева
    ax.set_yticks(y_ticks)
    ax.set_yticklabels(y_labels, fontsize=9, fontweight='bold')
    ax.tick_params(left=False, bottom=False, labelbottom=False)

    # Убираем внешнюю рамку
    for spine in ax.spines.values():
        spine.set_visible(False)

    ax.set_title(model_title, fontsize=15, fontweight='bold', pad=20)
    plt.tight_layout()

    if save_path:
        plt.savefig(save_path, bbox_inches='tight', dpi=600)

    plt.show()